# HMIE Batch Validation

Validate HMIE/Scale datasets using `datamaite`. Point this notebook at a
directory containing one or more batches and it will check:

- **Folder structure** -- snippet directories with `seq_*/` video containers
- **Video integrity** -- each video opens and decodes representative frames
- **Annotation coverage** -- every annotation pairs to a video (and vice-versa)
- **Scale spec compliance** -- annotation JSON conforms to the Scale Video Playback schema

Set `CHECK_VIDEO = False` below to skip video decoding for a faster JSON-only run.

In [ ]:
from pathlib import Path

import datamaite

print(f"datamaite {datamaite.__version__}")

# -- Configuration --------------------------------------------------------
# Point at a directory containing your datasets. This can be a single dataset
# (batch) directory, a directory of dataset directories, or a directory of
# grouped collections (e.g. valid/, bad-*/, warn-*/) -- the next cell
# discovers every dataset underneath either way.
BATCH_ROOT = Path("/path/to/hmie-mock-data/datasets")

# CHECK_VIDEO controls the per-file video probe (opening each .mp4 with
# OpenCV and decoding sample frames to detect corruption, dropped frames,
# wrong FPS, etc.). It does NOT control whether videos must exist:
# annotation/video pair discovery (orphan_annotation, orphan_video,
# multiple_videos_in_seq_mp4) and folder-structure checks always run.
#
# Set False for a fast JSON-only sweep (annotation schema + Scale spec
# compliance + folder layout). Set True for the full integrity scan.
CHECK_VIDEO = True

## Discover and validate every dataset

`find_batch_roots` locates the dataset (batch) directories under `BATCH_ROOT` --
a directory that directly contains snippet folders. When datasets are grouped
into sub-collections (for example `valid/`, `bad-corrupt-video/`,
`warn-fps-mismatch/`), we gather the batch roots from each sub-collection so a
single run covers *all* of them.

`validate_batches` then runs the full validation pipeline on each discovered
dataset and yields `(dataset_path, ValidationResult)` pairs as it goes -- so
the caller can stream results without buffering tens of thousands of datasets.

Each `ValidationResult` covers the four requirement checks:

- folder structure (snippet dirs, `seq_*/` containers)
- annotation coverage (every annotation paired to a video, no orphans)
- Scale spec compliance (annotation JSON against the Scale Video Playback schema)
- video integrity (per-file decode probe, only when `check_video_integrity=True`)

`result.summary()` prints a short pass/fail block per dataset; the result
object itself carries the full finding list, per-check counts, and label
histogram for downstream programmatic use.

In [ ]:
from datamaite import ValidationResult, find_batch_roots, validate_batches

# _batch_label is the same helper render_html_report_multi uses to name each
# dataset, so the console rollup, the collection report, and the optional
# per-dataset files all agree. It labels a dataset by its path relative to
# BATCH_ROOT (keeping grouped layouts distinct) and falls back to the leaf
# name when BATCH_ROOT *is* the dataset.
from datamaite._report import _batch_label

# Discover every dataset under BATCH_ROOT. find_batch_roots looks one level
# down; when datasets are grouped into sub-collections, gather the batch roots
# from each subdirectory too so one run covers the whole tree.
dataset_dirs = find_batch_roots(BATCH_ROOT)
if not dataset_dirs:
    dataset_dirs = sorted(batch for sub in BATCH_ROOT.iterdir() if sub.is_dir() for batch in find_batch_roots(sub))

results: list[tuple[Path, ValidationResult]] = []

for batch, result in validate_batches(dataset_dirs, check_video_integrity=CHECK_VIDEO):
    results.append((batch, result))
    print(f"--- {_batch_label(batch, BATCH_ROOT)} ---")
    print(result.summary())

print(f"Validated {len(results)} dataset(s)")

## Roll the results up

A short cross-batch summary: how many batches passed, total errors and
warnings, and a per-batch pass/fail line with snippet and finding counts.

In [ ]:
if not results:
    print("No datasets produced results.")
else:
    passed = sum(1 for _, r in results if r.passed)
    total_errs = sum(sum(r.finding_severity_counts["error"].values()) for _, r in results)
    total_warns = sum(sum(r.finding_severity_counts["warning"].values()) for _, r in results)
    print(f"{passed}/{len(results)} datasets passed")
    print(f"{total_errs:,} errors, {total_warns:,} warnings total\n")

    for batch, r in results:
        tag = "PASS" if r.passed else "FAIL"
        errs = sum(r.finding_severity_counts["error"].values())
        warns = sum(r.finding_severity_counts["warning"].values())
        name = _batch_label(batch, BATCH_ROOT)
        print(f"  {tag}  {name}  ({r.snippet_count:,} snippets, {errs:,} errors, {warns:,} warnings)")

## Generate an HTML report for the whole collection

For a shareable artifact (review, hand-off, attaching to a ticket),
`render_html_report_multi` rolls every batch into **one** self-contained HTML
file built for a collection of datasets:

- a landing view with an **"X of Y datasets passed"** headline and a **table
  with one row per dataset** -- snippet / annotation / error / warning counts
  and a PASS/FAIL pill -- so you can see *which datasets failed* at a glance;
- **click any dataset row** to drill into that dataset's requirement-check
  indicators, per-check finding counts, and label histogram -- *what failed on
  each dataset* -- with a breadcrumb back to the collection view.

Open the output path in a browser -- no server, no extra assets.

In [ ]:
from datamaite import render_html_report_multi

# One consolidated, self-contained report for the whole collection. The
# landing page lists every dataset (pass/fail + counts); clicking a row drills
# into that dataset's findings. render_html_report_multi returns a string, so
# the file write / output location is entirely up to the caller.
report_dir = Path("datamaite-reports")
report_dir.mkdir(exist_ok=True)

collection_report = report_dir / f"{BATCH_ROOT.name}-collection.html"
collection_report.write_text(render_html_report_multi(results, BATCH_ROOT), encoding="utf-8")

passed = sum(1 for _, r in results if r.passed)
print(f"Collection report: {collection_report}")
print(f"  {passed}/{len(results)} datasets passed -- open the file and click any dataset row to drill in")

### Optional: one report per dataset

The collection report above is usually what you want. If you instead need a
standalone file for a single dataset (e.g. to hand one batch to its owner),
`render_html_report` renders any one `ValidationResult` on its own.

In [ ]:
from datamaite import render_html_report

# Optional: a standalone HTML file per dataset, alongside the collection report.
# Flatten the dataset's collection label into the filename so datasets that
# share a leaf name across sub-collections do not overwrite each other. Using
# _batch_label means a single-dataset BATCH_ROOT writes "<dataset>.html", not
# the "..html" a bare relative_to(".") would produce.
for batch, result in results:
    name = _batch_label(batch, BATCH_ROOT).replace("/", "__")
    out = report_dir / f"{name}.html"
    out.write_text(render_html_report(result), encoding="utf-8")
    print(f"  {out}")